# Proyecto MOTHER: Motor Operativo de Triage Hospitalario
## Fase 1: Ingesta, Limpieza e Integración de Datos

Este notebook realiza la carga de los datasets originales de Víctimas y Hechos.  
Se encargará de la limpieza inicial, estandarización, manejo adecuado de valores nulos y duplicados, y finalmente realizará el merge para crear el dataset integrado que se usará en etapas posteriores.

In [1]:
# 1_preparacion_datos.ipynb - Bloque 1: Importaciones y configuración de logging

import logging
import pandas as pd
import numpy as np

# Configuración de logging: guardar todo en 'mother_pipeline.log'
logging.basicConfig(
    filename='mother_pipeline.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    filemode='w'
)
logger = logging.getLogger()

logger.info("Inicio del notebook 1_Preparacion_Datos")


### 1.1 Definición de Funciones de Limpieza y Estandarización
Para garantizar la reproducibilidad y mantener un código limpio, definimos funciones específicas para la inspección de valores categóricos y la normalización de textos. Esto nos permitirá tratar inconsistencias semánticas (ej. "MISSING" vs "SD") de forma sistemática.

In [2]:
# 1_preparacion_datos.ipynb - Bloque 2: Funciones de estandarización y limpieza para Dataset Víctimas

def mostrar_valores_unicos(df, columnas):
    """
    Imprime los valores únicos presentes en las columnas categóricas especificadas, incluyendo los nulos.
    """
    logger.info("--- INSPECCIÓN DE VALORES ÚNICOS EN COLUMNAS CATEGÓRICAS ---")
    for col in columnas:
        valores_distintos = df[col].unique()
        logger.info(f"Valores únicos para la columna '{col}': {list(valores_distintos)}")

def estandarizar_registros(df, columnas_categoricas):
    """
    Convierte todos los valores de las columnas categóricas a mayúsculas y elimina espacios extra para evitar duplicados como 'moto' y 'MOTO'.
    También reemplaza cadenas que representan nulos por np.nan.
    """
    df_clean = df.copy()
    for col in columnas_categoricas:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].astype(str).str.upper().str.strip()
            variantes_nulos = ['NAN', 'NONE', '<NA>', 'NULL', 'ND', 'SD', '']
            df_clean[col] = df_clean[col].replace(variantes_nulos, np.nan)
    return df_clean

def detectar_valores_sd(df):
    """
    Cuenta la cantidad de valores 'SD' en columnas de tipo texto y los reporta.
    """
    logger.info("--- Conteo de valores 'SD' por columna ---")
    sd_totales = 0
    for column in df.select_dtypes(include=['object']).columns:
        sd_count = (df[column] == 'SD').sum()
        if sd_count > 0:
            logger.info(f"Columna '{column}': {sd_count} registros con 'SD'")
            sd_totales += sd_count
    if sd_totales == 0:
        logger.info("No se encontraron valores 'SD'.")
    logger.info("-" * 40)

def identificar_duplicados(df, columna_id):
    """
    Identifica y muestra registros duplicados basados en una columna.
    """
    duplicados = df[df.duplicated(subset=[columna_id], keep=False)]
    if not duplicados.empty:
        logger.info(f"--- Se encontraron {len(duplicados)} registros con '{columna_id}' duplicado ---")
        logger.info(duplicados.sort_values(by=columna_id).head())
    else:
        logger.info(f"--- No hay valores duplicados en la columna '{columna_id}' ---")

def estandarizar_nulos_y_tipos(df):
    """
    Reemplaza textos representativos de nulos 'SD', y convierte columnas 'edad_victima' a numérica y 'fecha_siniestro' a datetime.
    """
    cols_texto = df.select_dtypes(include=['object', 'string']).columns
    variantes_sd = ['SD', 'SD ', ' SD', 'sd', 'Sd']
    df[cols_texto] = df[cols_texto].replace(variantes_sd, np.nan)
    if 'edad_victima' in df.columns:
        df['edad_victima'] = pd.to_numeric(df['edad_victima'], errors='coerce')
    if 'fecha_siniestro' in df.columns:
        df['fecha_siniestro'] = pd.to_datetime(df['fecha_siniestro'], errors='coerce')
    return df

def eliminar_duplicados_exactos(df):
    """
    Elimina registros donde toda la fila es exactamente igual a otra, manteniendo solo la primera aparición.
    """
    filas_antes = len(df)
    df.drop_duplicates(inplace=True)
    filas_despues = len(df)
    eliminados = filas_antes - filas_despues
    if eliminados > 0:
        logger.info(f"Éxito: Se eliminaron {eliminados} filas exactamente duplicadas.")
        logger.info(f"El dataset se redujo de {filas_antes} a {filas_despues} registros.")
    else:
        logger.info("No se encontraron duplicados exactos para eliminar.")


In [3]:
from pathlib import Path
import urllib.request

BASE_DIR = Path.cwd().parent
RAW_DIR = BASE_DIR / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

ruta_hechos = RAW_DIR / 'siniestros_viales_hechos.csv'
ruta_victimas = RAW_DIR / 'siniestros_viales_victimas.csv'

def descargar_si_no_existe(ruta_local, url_descarga):
    if not ruta_local.exists():
        try:
            logger.info(f"Archivo {ruta_local} no existe. Descargando desde {url_descarga}...")
            urllib.request.urlretrieve(str(url_descarga), str(ruta_local))
            logger.info(f"Descarga exitosa: {ruta_local}")
        except Exception as e:
            logger.error(f"No se pudo descargar {ruta_local}: {e}")
            raise FileNotFoundError(f"Debe colocar manualmente {ruta_local} en la carpeta 'data/raw'.")

url_hechos = 'https://drive.google.com/uc?export=download&id=1TzZuUW45mPMPTR1auG7FPj_4a6AB4dc_'
url_victimas = 'https://drive.google.com/uc?export=download&id=1X-Qdj99nvePum4f5y7URt5PlPjRSS5gJ'

descargar_si_no_existe(ruta_hechos, url_hechos)
descargar_si_no_existe(ruta_victimas, url_victimas)

df_hechos = pd.read_csv(ruta_hechos)
df_victimas = pd.read_csv(ruta_victimas)

logger.info(f"Dataset 'hechos' cargado con {len(df_hechos)} registros y {len(df_hechos.columns)} columnas.")
logger.info(f"Dataset 'víctimas' cargado con {len(df_victimas)} registros y {len(df_victimas.columns)} columnas.")


In [4]:
# 1_preparacion_datos.ipynb - Bloque 4: Ejecución - Transformaciones y limpieza inicial


# Estandarización de nombres de columnas a minúsculas y sin espacios en Víctimas (como en tu código original)
df_victimas.columns = df_victimas.columns.str.lower().str.strip()
logger.info("Nombres de columnas de df_victimas estandarizados a minúsculas y sin espacios.")

# Define las columnas categóricas de Victimas para limpieza
cols_cat_victimas = ['gravedad_victima', 'sexo_victima', 'rol_victima', 'modo_desplazamiento_victima']

# Aplicar estandarización tipográfica sobre Victimas
df_victimas = estandarizar_registros(df_victimas, cols_cat_victimas)
logger.info("Estandarización tipográfica aplicada en dataset Víctimas.")

# Detectar valores 'SD' en Víctimas
detectar_valores_sd(df_victimas)

# Convertir tipos y estandarizar nulos para Víctimas
df_victimas = estandarizar_nulos_y_tipos(df_victimas)
logger.info("Conversión de tipos y estandarización de nulos realizada en dataset Víctimas.")

# Eliminar duplicados exactos en Víctimas
eliminar_duplicados_exactos(df_victimas)


In [5]:
# 1_preparacion_datos.ipynb - Bloque 5: Limpieza, estandarización y control en dataset Hechos

# Estandarizamos nombres de columnas a minúsculas y sin espacios en Hechos
df_hechos.columns = df_hechos.columns.str.lower().str.strip()
logger.info("Nombres de columnas de df_hechos estandarizados a minúsculas y sin espacios.")

# Definimos columnas categóricas de Hechos para limpieza y estandarización
cols_cat_hechos = [
    'comuna_siniestro', 'tipo_de_via_siniestro', 'participantes_siniestro',
    'modo_desplazamiento_victima', 'contraparte_siniestro', 'gravedad_siniestro'
]

# Reemplazamos 'av. gral. paz' por 'autopista' en tipo_de_via_siniestro, conforme tu código original
df_hechos['tipo_de_via_siniestro'] = df_hechos['tipo_de_via_siniestro'].replace('AV. GRAL. PAZ', 'AUTOPISTA')

# Aplicar estandarización tipográfica en Hechos
df_hechos = estandarizar_registros(df_hechos, cols_cat_hechos)
logger.info("Estandarización tipográfica aplicada en dataset Hechos.")

# Detectar valores 'SD' en Hechos
detectar_valores_sd(df_hechos)

# Convertir tipos y estandarizar nulos para Hechos
df_hechos = estandarizar_nulos_y_tipos(df_hechos)
logger.info("Conversión de tipos y estandarización de nulos realizada en dataset Hechos.")

# Identificar duplicados por id_siniestro en Hechos
identificar_duplicados(df_hechos, 'id_siniestro')


In [6]:
from pathlib import Path

logger.info("Iniciando merge de datasets Víctimas y Hechos por 'id_siniestro'...")

# Unificamos formatos en columnas clave para evitar errores en merge
df_victimas['id_siniestro'] = df_victimas['id_siniestro'].astype(str).str.upper().str.strip()
df_hechos['id_siniestro'] = df_hechos['id_siniestro'].astype(str).str.upper().str.strip()

df_victimas['fecha_siniestro'] = pd.to_datetime(df_victimas['fecha_siniestro'], errors='coerce')
df_hechos['fecha_siniestro'] = pd.to_datetime(df_hechos['fecha_siniestro'], errors='coerce')

# Realizamos merge left: mantenemos todas las víctimas y agregamos contexto de hechos
df_integrado = pd.merge(df_victimas, df_hechos.drop(columns=['gravedad_siniestro']), on=['id_siniestro', 'fecha_siniestro'], how='left')

logger.info(f"Merge completo. Dataset integrado con {df_integrado.shape[0]} filas y {df_integrado.shape[1]} columnas.")

# Definir ruta absoluta para data/processed desde la raíz del proyecto
BASE_DIR = Path.cwd().parent
processed_dir = BASE_DIR / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

output_path = processed_dir / 'datos_integrados.csv'

# Exportamos a CSV en la ruta absoluta correcta
df_integrado.to_csv(output_path, index=False)
logger.info(f"Archivo '{output_path}' exportado correctamente.")


In [7]:
# 1_preparacion_datos.ipynb - Bloque 7: Checkpoints rápidos para verificación post-proceso

logger.info("Iniciando checkpoints de validación rápida del dataset integrado.")

# Mostrar forma final
logger.info(f"Forma final del dataset integrado: {df_integrado.shape}")

# Conteo de valores nulos por columna
nulos = df_integrado.isna().sum().sort_values(ascending=False)
logger.info(f"Top columnas con más valores nulos en dataset integrado:\n{nulos[nulos > 0].head(10)}")

# Mostrar valores únicos en columnas clave para validar calidad
columnas_clave = ['gravedad_victima', 'sexo_victima', 'rol_victima', 'modo_desplazamiento_victima', 'tipo_de_via_siniestro']

for col in columnas_clave:
    if col in df_integrado.columns:
        uniques = df_integrado[col].dropna().unique()
        logger.info(f"Valores únicos en '{col}': {uniques}")

logger.info("Checkpoints post-proceso completados.")
